# Fetch + filter Nasdaq ITCH days (runs in Colab)

Downloads full-day TotalView-ITCH 5.0 files from Nasdaq's public server and keeps only what
the project needs, in your Google Drive:
* `MyDrive/lobgen/filtered/<date>.itch.gz`: the stocks (GOOG, INTC), full order flow.
* `MyDrive/lobgen/filtered_market/<date>.itch.gz`: the market ETFs (QQQ, SMH), and from them
  `MyDrive/lobgen/market/<ETF>_<date>.npz`: best bid and ask after every change. This is how
  news reaches the model (src/lobgen/market.py).

Raw files are deleted after filtering. A day is downloaded only if something is missing for it,
so with the stock files already in Drive this run fetches only the ETFs (~15 min per day: ~10
download, ~5 filter), then builds the quote series (a few minutes per ETF-day).

**Run:** Runtime -> Run all (a CPU runtime is enough). Approve the Drive prompt. Keep the tab
open until the last cell prints `done`. Safe to re-run: finished work is skipped.

In [ ]:
DAYS = {  # date -> file on emi.nasdaq.com
    "2025-12-08": "S120825-v50.txt.gz",
    "2025-12-09": "S120925-v50.txt.gz",
    "2025-12-10": "S121025-v50.txt.gz",
    "2025-12-11": "S121125-v50.txt.gz",
    "2025-12-12": "S121225-v50.txt.gz",
    # "2026-06-12": "S061226-v50.txt.gz",  # optional: out-of-time test day, 17.9 GB
}
TICKERS = ["GOOG", "INTC"]
MARKET = ["QQQ", "SMH"]          # must match lobgen.market.ETFS
BASE = "https://emi.nasdaq.com/ITCH/Nasdaq%20ITCH/"
DRIVE = "/content/drive/MyDrive/lobgen"
OUT = f"{DRIVE}/filtered"
OUT_MKT = f"{DRIVE}/filtered_market"
MKT = f"{DRIVE}/market"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
for d in (OUT, OUT_MKT, MKT, "/content/pkg/lobgen"):
    os.makedirs(d, exist_ok=True)
!pip -q install numba sortedcontainers

In [ ]:
%%writefile /content/pkg/lobgen/__init__.py
"""lobgen: Nasdaq ITCH -> LOBSTER conversion and synthetic order book generation."""

In [ ]:
%%writefile /content/pkg/lobgen/itch.py
"""Streaming reader/filter for Nasdaq TotalView-ITCH 5.0 binary files.

Wire format: each message is a 2-byte big-endian length followed by that many
bytes. Byte 0 of the message is the type char, bytes 1-2 the stock locate code,
3-4 tracking number, 5-10 a 6-byte nanoseconds-since-midnight timestamp.

A full-day file is 4-18 GB gzipped and holds every Nasdaq symbol, so `filter_file`
makes one streaming pass and keeps only the system-event ('S') messages plus
messages whose locate code belongs to a requested ticker. The output is itself a
valid ITCH stream, small enough to parse in pure Python.
"""

from __future__ import annotations

import gzip
import struct
import sys
import time
from pathlib import Path
from typing import Iterator

import numpy as np
from numba import njit

# Message body lengths (excluding the 2-byte length prefix), ITCH 5.0 spec.
MSG_LEN = {
    b"S": 12, b"R": 39, b"H": 25, b"Y": 20, b"L": 26, b"V": 35, b"W": 12,
    b"K": 28, b"J": 35, b"h": 21, b"A": 36, b"F": 40, b"E": 31, b"C": 36,
    b"X": 23, b"D": 19, b"U": 35, b"P": 44, b"Q": 40, b"B": 19, b"I": 50,
    b"N": 20, b"O": 48,
}

_S = ord("S")
_R = ord("R")


@njit(cache=True)
def _scan(buf, n, want_names, locate_map, out):
    """Walk length-prefixed messages in buf[:n]; copy kept ones into out.

    Returns (bytes_consumed, bytes_written, messages_seen). Stops at the last
    complete message so the caller can carry the partial tail to the next chunk.
    locate_map is updated in place when a wanted ticker's 'R' message is seen.
    """
    pos = 0
    w = 0
    seen = 0
    k_names = want_names.shape[0]
    while pos + 2 <= n:
        length = (np.int64(buf[pos]) << 8) | np.int64(buf[pos + 1])
        end = pos + 2 + length
        if end > n:
            break
        t = buf[pos + 2]
        loc = (np.int64(buf[pos + 3]) << 8) | np.int64(buf[pos + 4])
        keep = False
        if t == _S:
            keep = True
        elif t == _R:
            # Stock symbol: 8 bytes at message offset 11 (+2 for prefix).
            for k in range(k_names):
                match = True
                for j in range(8):
                    if buf[pos + 13 + j] != want_names[k, j]:
                        match = False
                        break
                if match:
                    locate_map[loc] = k
                    keep = True
                    break
        elif locate_map[loc] >= 0:
            keep = True
        if keep:
            for i in range(pos, end):
                out[w] = buf[i]
                w += 1
        pos = end
        seen += 1
    return pos, w, seen


def _pad_symbol(sym: str) -> bytes:
    b = sym.encode("ascii")
    if len(b) > 8:
        raise ValueError(f"symbol too long: {sym}")
    return b.ljust(8, b" ")


def filter_file(
    raw_path: str | Path,
    tickers: list[str],
    out_path: str | Path,
    chunk_bytes: int = 1 << 27,
    allow_truncated: bool = False,
    log_every_s: float = 30.0,
) -> dict:
    """Stream a gzipped ITCH file (or '-' for gzipped stdin), keep 'S' + messages for `tickers`.

    Reading stdin lets a remote box run `curl URL | lobgen filter - ...` without storing
    the raw multi-GB day.
    """
    from_stdin = str(raw_path) == "-"
    out_path = Path(out_path)
    want = np.frombuffer(b"".join(_pad_symbol(t) for t in tickers), dtype=np.uint8)
    want = want.reshape(len(tickers), 8).copy()
    locate_map = np.full(1 << 16, -1, dtype=np.int64)

    buf = np.empty(chunk_bytes, dtype=np.uint8)
    out = np.empty(chunk_bytes, dtype=np.uint8)
    mv = memoryview(buf)
    have = 0
    total_seen = total_kept_bytes = total_raw_bytes = 0
    truncated = False
    compressed_size = None if from_stdin else Path(raw_path).stat().st_size
    t0 = last_log = time.time()
    src = gzip.GzipFile(fileobj=sys.stdin.buffer) if from_stdin else gzip.open(raw_path, "rb")

    with src as f, open(out_path, "wb") as o:
        while True:
            try:
                n = f.readinto(mv[have:])
            except EOFError:
                if not allow_truncated:
                    raise
                truncated, n = True, 0
            have += n
            total_raw_bytes += n
            consumed, w, seen = _scan(buf, have, want, locate_map, out)
            o.write(memoryview(out)[:w])
            total_seen += seen
            total_kept_bytes += w
            rem = have - consumed
            if consumed == 0 and have == chunk_bytes:
                raise ValueError(f"message longer than chunk_bytes={chunk_bytes}")
            if rem:
                buf[:rem] = buf[consumed:have].copy()
            have = rem
            now = time.time()
            if now - last_log > log_every_s:
                pct = f"{f.fileobj.tell() / compressed_size:6.1%} compressed read | " if compressed_size else ""
                print(
                    f"  {pct}{total_raw_bytes / 1e9:7.2f} GB raw | {total_seen / 1e6:8.1f} M msgs | "
                    f"{(now - t0) / 60:5.1f} min",
                    file=sys.stderr, flush=True,
                )
                last_log = now
            if n == 0:
                break

    if have and not truncated:
        raise ValueError(f"{have} trailing bytes do not form a complete message")
    found = {t: int(np.flatnonzero(locate_map == k)[0]) if (locate_map == k).any() else None
             for k, t in enumerate(tickers)}
    missing = [t for t, loc in found.items() if loc is None]
    if missing:
        raise ValueError(f"tickers not in stock directory: {missing}")
    return {
        "messages_seen": total_seen,
        "raw_bytes": total_raw_bytes,
        "kept_bytes": total_kept_bytes,
        "locates": found,
        "truncated": truncated,
        "minutes": (time.time() - t0) / 60,
    }


# ---------------------------------------------------------------------------
# Pure-Python message decoding for the (small) filtered stream.
# ---------------------------------------------------------------------------

def load_stream(path: str | Path) -> bytes:
    """Read a filtered ITCH file (.gz ok) into memory."""
    p = Path(path)
    return gzip.decompress(p.read_bytes()) if p.suffix == ".gz" else p.read_bytes()


def iter_messages(src: str | Path | bytes) -> Iterator[bytes]:
    """Yield raw message bodies (without length prefix) from a filtered stream or its path."""
    data = src if isinstance(src, (bytes, bytearray)) else load_stream(src)
    pos, n = 0, len(data)
    while pos + 2 <= n:
        (length,) = struct.unpack_from(">H", data, pos)
        end = pos + 2 + length
        if end > n:
            raise ValueError(f"truncated message at byte {pos}")
        yield data[pos + 2:end]
        pos = end


def header(msg: bytes) -> tuple[str, int, int]:
    """(type char, locate, timestamp_ns)."""
    t = chr(msg[0])
    locate = (msg[1] << 8) | msg[2]
    ts = int.from_bytes(msg[5:11], "big")
    return t, locate, ts


# Body decoders, offsets after the 11-byte common header.
_U64 = struct.Struct(">Q")
_ADD = struct.Struct(">QcI8sI")          # ref, side, shares, stock, price
_EXEC = struct.Struct(">QIQ")            # ref, shares, match
_EXEC_PX = struct.Struct(">QIQcI")       # ref, shares, match, printable, price
_CANCEL = struct.Struct(">QI")           # ref, cancelled shares
_REPLACE = struct.Struct(">QQII")        # orig ref, new ref, shares, price
_TRADE = struct.Struct(">QcI8sIQ")       # ref, side, shares, stock, price, match
_CROSS = struct.Struct(">Q8sIQc")        # shares, stock, price, match, cross type
_HALT = struct.Struct(">8scc4s")         # stock, state, reserved, reason
_DIR = struct.Struct(">8scc")            # stock, market category, FS status


def decode(msg: bytes):
    """Decode one message into (type, locate, ts, fields tuple). Unused types -> fields None."""
    t, locate, ts = header(msg)
    if t in ("A", "F"):
        f = _ADD.unpack_from(msg, 11)
    elif t == "E":
        f = _EXEC.unpack_from(msg, 11)
    elif t == "C":
        f = _EXEC_PX.unpack_from(msg, 11)
    elif t == "X":
        f = _CANCEL.unpack_from(msg, 11)
    elif t == "D":
        f = _U64.unpack_from(msg, 11)
    elif t == "U":
        f = _REPLACE.unpack_from(msg, 11)
    elif t == "P":
        f = _TRADE.unpack_from(msg, 11)
    elif t == "Q":
        f = _CROSS.unpack_from(msg, 11)
    elif t == "H":
        f = _HALT.unpack_from(msg, 11)
    elif t == "R":
        f = _DIR.unpack_from(msg, 11)
    elif t == "S":
        f = (msg[11:12],)
    else:
        f = None
    return t, locate, ts, f

In [ ]:
%%writefile /content/pkg/lobgen/book.py
"""Order-level limit order book. Prices are integer dollars x 10000 (ITCH == LOBSTER units)."""

from __future__ import annotations

from sortedcontainers import SortedDict

BUY, SELL = 1, -1
EMPTY_ASK = 9_999_999_999
EMPTY_BID = -9_999_999_999


class BookError(RuntimeError):
    pass


class OrderBook:
    def __init__(self) -> None:
        self.orders: dict[int, list[int]] = {}  # ref -> [side, price, shares]
        self.bids: SortedDict = SortedDict()     # price -> total shares
        self.asks: SortedDict = SortedDict()

    def _levels(self, side: int) -> SortedDict:
        return self.bids if side == BUY else self.asks

    def add(self, ref: int, side: int, price: int, shares: int) -> None:
        if ref in self.orders:
            raise BookError(f"duplicate order ref {ref}")
        if shares <= 0:
            raise BookError(f"non-positive add size {shares} for ref {ref}")
        self.orders[ref] = [side, price, shares]
        lv = self._levels(side)
        lv[price] = lv.get(price, 0) + shares

    def reduce(self, ref: int, shares: int) -> tuple[int, int, int]:
        """Remove `shares` from an order. Returns (side, price, remaining)."""
        o = self.orders.get(ref)
        if o is None:
            raise BookError(f"unknown order ref {ref}")
        side, price, have = o
        if shares <= 0 or shares > have:
            raise BookError(f"reduce {shares} from ref {ref} holding {have}")
        remaining = have - shares
        lv = self._levels(side)
        left = lv[price] - shares
        if left:
            lv[price] = left
        else:
            del lv[price]
        if remaining:
            o[2] = remaining
        else:
            del self.orders[ref]
        return side, price, remaining

    def delete(self, ref: int) -> tuple[int, int, int]:
        """Remove an order entirely. Returns (side, price, shares removed)."""
        o = self.orders.get(ref)
        if o is None:
            raise BookError(f"unknown order ref {ref}")
        side, price, shares = o
        self.reduce(ref, shares)
        return side, price, shares

    def best_bid(self) -> int | None:
        return self.bids.peekitem(-1)[0] if self.bids else None

    def best_ask(self) -> int | None:
        return self.asks.peekitem(0)[0] if self.asks else None

    def snapshot(self, levels: int) -> list[int]:
        """LOBSTER orderbook row: askP1, askV1, bidP1, bidV1, askP2, ... (4 * levels ints)."""
        row: list[int] = []
        ak, av = self.asks.keys(), self.asks
        bk, bv = self.bids.keys(), self.bids
        na, nb = len(ak), len(bk)
        for i in range(levels):
            if i < na:
                p = ak[i]
                row += (p, av[p])
            else:
                row += (EMPTY_ASK, 0)
            if i < nb:
                p = bk[nb - 1 - i]
                row += (p, bv[p])
            else:
                row += (EMPTY_BID, 0)
        return row

In [ ]:
%%writefile /content/pkg/lobgen/lobster.py
"""Convert a filtered ITCH 5.0 stream into LOBSTER-format message + orderbook files.

LOBSTER message columns: time (s after midnight), type, order id, size, price, direction.
  type 1 submit | 2 partial cancel | 3 delete | 4 visible execution | 5 hidden execution
       6 cross trade | 7 trading halt
  direction: 1 buy limit order, -1 sell limit order (an execution's direction is the
  resting order's side, so a type-4 with direction -1 is a buyer-initiated trade).
  price: integer dollars x 10000 (same as ITCH's 4 implied decimals; never converted to float).

Conversion choices (documented because they change the statistics):
  * The book is rebuilt from the first message of the file so pre-market orders exist
    at the open; only events with start <= t < end are written.
  * ITCH 'U' (replace) -> type 3 delete of the old ref, then type 1 submit of the new ref,
    same timestamp, side inherited from the old order.
  * ITCH 'C' (execution with price) -> type 4 at the resting order's LIMIT price, so every
    type-4 row reduces the book level at its own price. The actual print price differs;
    counts of such rows are in the report.
  * ITCH 'P' (hidden execution) -> type 5, order id 0. Its side flag is unreliable: Nasdaq's
    spec says it has always been 'B' since 2014-07-14. Do not use type-5 direction.
  * ITCH 'Q' (cross) -> type 6, order id 0, direction -1. Cross executions of displayed
    orders also arrive as non-printable 'E'/'C' rows (type 4), so cross volume appears twice.
  * ITCH 'H' state H (halt) -> type 7 price -1; Q (quote only) -> price 0; T (trading) -> 1.
  * Empty book levels: ask 9999999999 / bid -9999999999, size 0.
"""

from __future__ import annotations

from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from .book import BUY, SELL, OrderBook
from .itch import decode, header, iter_messages, load_stream

SUBMIT, CANCEL, DELETE, EXEC, HIDDEN, CROSS, HALT = 1, 2, 3, 4, 5, 6, 7
NS = 1_000_000_000
_SIDE = {b"B": BUY, b"S": SELL}
_HALT_PRICE = {b"H": -1, b"P": -1, b"Q": 0, b"T": 1}


class _Rows:
    """Preallocated row buffer (a full GOOG day is ~9M rows x 40 cols; doubling would need ~8 GB)."""

    def __init__(self, n: int, width: int) -> None:
        self.a = np.empty((n, width), dtype=np.int64)
        self.n = 0

    def append(self, row) -> None:
        self.a[self.n] = row
        self.n += 1

    def array(self) -> np.ndarray:
        if self.n != len(self.a):
            raise RuntimeError(f"event count mismatch: counted {len(self.a)}, emitted {self.n}")
        return self.a


def _count_events(data: bytes, sym: bytes, start_ns: int, end_ns: int) -> int:
    """Number of LOBSTER rows convert() will emit (a replace 'U' emits two)."""
    locate, n = None, 0
    for raw in iter_messages(data):
        t, loc, ts = header(raw)
        if t == "R" and raw[11:19] == sym:
            locate = loc
        elif loc == locate and start_ns <= ts < end_ns and t in _EMITTING:
            n += 2 if t == "U" else 1
    return n


_EMITTING = frozenset("AFECXDUPQH")


@dataclass
class Report:
    ticker: str
    date: str
    locate: int | None = None
    itch_counts: Counter = field(default_factory=Counter)
    lobster_counts: Counter = field(default_factory=Counter)
    exec_with_price: int = 0
    exec_with_price_nonprintable: int = 0
    exec_with_price_differs: int = 0
    crossed_rows: int = 0
    first_crossed_s: float | None = None
    last_crossed_s: float | None = None
    open_orders_at_end: int = 0
    system_events: list = field(default_factory=list)

    def as_dict(self) -> dict:
        d = dict(self.__dict__)
        d["itch_counts"] = dict(sorted(self.itch_counts.items()))
        d["lobster_counts"] = {int(k): v for k, v in sorted(self.lobster_counts.items())}
        return d


def convert(
    filtered_path: str | Path,
    ticker: str,
    date: str,
    levels: int = 10,
    start_s: int = 34_200,
    end_s: int = 57_600,
) -> tuple[np.ndarray, np.ndarray, Report]:
    """Returns (messages[N,6] with time in ns, book[N, 4*levels], report)."""
    start_ns, end_ns = start_s * NS, end_s * NS
    sym = ticker.encode().ljust(8, b" ")
    data = load_stream(filtered_path)
    n_rows = _count_events(data, sym, start_ns, end_ns)
    book = OrderBook()
    msgs, snaps = _Rows(n_rows, 6), _Rows(n_rows, 4 * levels)
    rep = Report(ticker=ticker, date=date)

    def emit(ts: int, typ: int, ref: int, size: int, price: int, direction: int) -> None:
        if not (start_ns <= ts < end_ns):
            return
        msgs.append((ts, typ, ref, size, price, direction))
        row = book.snapshot(levels)
        snaps.append(row)
        rep.lobster_counts[typ] += 1
        ask, bid = row[0], row[2]
        if row[1] and row[3] and bid >= ask:
            rep.crossed_rows += 1
            s = ts / NS
            rep.first_crossed_s = s if rep.first_crossed_s is None else rep.first_crossed_s
            rep.last_crossed_s = s

    _replay(data, sym, book, emit, rep)
    if rep.locate is None:
        raise ValueError(f"{ticker} not found in stock directory of {filtered_path}")
    rep.open_orders_at_end = len(book.orders)
    return msgs.array(), snaps.array(), rep


def preopen_orders(filtered_path: str | Path, ticker: str, start_s: int = 34_200) -> np.ndarray:
    """Order-level book just before the first `ticker` message at or after start_s.

    Returns int64 [K, 4] rows (ref, side, price, shares) in time priority (the order rows
    entered the book; a replace re-enters at the back). With the LOBSTER messages from
    start_s onward this is enough to rebuild the full order-level book at any later row.
    """
    sym = ticker.encode().ljust(8, b" ")
    book = OrderBook()
    rep = Report(ticker=ticker, date="")
    _replay(load_stream(filtered_path), sym, book, lambda *a: None, rep, stop_ns=start_s * NS)
    if rep.locate is None:
        raise ValueError(f"{ticker} not found in stock directory of {filtered_path}")
    rows = [(ref, side, price, shares) for ref, (side, price, shares) in book.orders.items()]
    return np.array(rows, dtype=np.int64).reshape(-1, 4)


def _replay(data: bytes, sym: bytes, book: OrderBook, emit, rep: Report,
            stop_ns: int | None = None) -> None:
    """Apply every message of `sym` to `book`, calling emit(...) once per LOBSTER row.

    With stop_ns, returns just before the first message of `sym` with timestamp >= stop_ns.
    """
    locate = None
    for raw in iter_messages(data):
        t, loc, ts, f = decode(raw)
        if t == "S":
            rep.system_events.append((f[0].decode(), ts / NS))
            continue
        if t == "R":
            if f[0] == sym:
                locate = loc
                rep.locate = loc
            continue
        if loc != locate or locate is None:
            continue
        if stop_ns is not None and ts >= stop_ns:
            return
        rep.itch_counts[t] += 1

        if t in ("A", "F"):
            ref, side, shares, _stock, price = f[:5]
            d = _SIDE[side]
            book.add(ref, d, price, shares)
            emit(ts, SUBMIT, ref, shares, price, d)
        elif t == "E":
            ref, shares, _match = f
            d, price, _ = book.reduce(ref, shares)
            emit(ts, EXEC, ref, shares, price, d)
        elif t == "C":
            ref, shares, _match, printable, px = f
            d, price, _ = book.reduce(ref, shares)
            rep.exec_with_price += 1
            rep.exec_with_price_nonprintable += printable == b"N"
            rep.exec_with_price_differs += px != price
            emit(ts, EXEC, ref, shares, price, d)
        elif t == "X":
            ref, shares = f
            d, price, _ = book.reduce(ref, shares)
            emit(ts, CANCEL, ref, shares, price, d)
        elif t == "D":
            (ref,) = f
            d, price, shares = book.delete(ref)
            emit(ts, DELETE, ref, shares, price, d)
        elif t == "U":
            old, new, shares, price = f
            d, old_price, old_shares = book.delete(old)
            emit(ts, DELETE, old, old_shares, old_price, d)
            book.add(new, d, price, shares)
            emit(ts, SUBMIT, new, shares, price, d)
        elif t == "P":
            _ref, side, shares, _stock, price, _match = f
            emit(ts, HIDDEN, 0, shares, price, _SIDE[side])
        elif t == "Q":
            shares, _stock, price, _match, _ctype = f
            emit(ts, CROSS, 0, shares, price, -1)
        elif t == "H":
            _stock, state, _res, _reason = f
            emit(ts, HALT, 0, 0, _HALT_PRICE.get(state, -1), -1)


def lobster_stem(ticker: str, date: str, start_s: int, end_s: int) -> str:
    return f"{ticker}_{date}_{start_s * 1000}_{end_s * 1000}"


def format_time(ts_ns: np.ndarray) -> np.ndarray:
    """Exact 'seconds.nanoseconds' strings; no float rounding."""
    sec, ns = np.divmod(ts_ns, NS)
    return np.char.add(np.char.add(sec.astype(str), "."), np.char.zfill(ns.astype(str), 9))


def write_lobster(
    msgs: np.ndarray, book: np.ndarray, out_dir: str | Path, ticker: str, date: str,
    levels: int, start_s: int = 34_200, end_s: int = 57_600, chunk: int = 500_000,
    csv: bool = True,
) -> list[Path]:
    """Always writes a compressed .npz; LOBSTER csv pair only if csv=True (a full day is ~2.4 GB)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = lobster_stem(ticker, date, start_s, end_s)
    npath = out_dir / f"{stem}_{levels}.npz"
    np.savez_compressed(npath, messages=msgs, book=book, levels=levels)
    if not csv:
        return [npath]
    mpath = out_dir / f"{stem}_message_{levels}.csv"
    bpath = out_dir / f"{stem}_orderbook_{levels}.csv"
    write_lobster_csv(msgs, book, mpath, bpath, chunk)
    return [npath, mpath, bpath]


def write_lobster_csv(msgs: np.ndarray, book: np.ndarray, mpath: Path, bpath: Path,
                      chunk: int = 500_000) -> None:
    with open(mpath, "w") as fm, open(bpath, "w") as fb:
        for i in range(0, len(msgs), chunk):
            m = msgs[i:i + chunk]
            df = pd.DataFrame(m[:, 1:], copy=False)
            df.insert(0, "time", format_time(m[:, 0]))
            df.to_csv(fm, header=False, index=False)
            pd.DataFrame(book[i:i + chunk], copy=False).to_csv(fb, header=False, index=False)

In [ ]:
%%writefile /content/pkg/lobgen/market.py
"""The market as an input: how news reaches a single stock's order book.

A tariff headline, a rate decision or a sector sell-off does not arrive in GOOG's order book as
text. It arrives as a move in the whole market, and market makers in GOOG re-price off index
futures and ETFs within milliseconds. So the channel this module gives the model is the market
itself: the Nasdaq quotes of QQQ (Nasdaq-100) and SMH (semiconductors, for INTC), taken from
the same ITCH files as the stocks, turned into clock-time features at every stock event.

Per ETF and per time scale tau in CLOCK_TAUS_S (0.1, 1, 10, 100 s):
    dev_tau(t) = log mid(t) - EMA_tau(log mid)(t)        in bps / 10, clipped to +-5
a continuous-time exponential average, so dev_tau ~ the ETF's return over the last ~tau; and
per tau in VOL_TAUS_S (1, 30 s) a turbulence measure:
    vol_tau(t) = log1p( sum_k |d log mid_k| e^{-(t - t_k)/tau} in bps ) / 3
The same clock-time deviations of the stock's *own* mid (CLOCK_TAUS_S) are carried in the
feature stream (features.py), so the model can see the gap between how far the market moved and
how far the stock has followed: lead-lag catch-up is what makes a stock "go down as much as it
should" after market-wide news.

Lookups are exact and causal: features at time t use only ETF quotes at or before t. During a
rollout the generated event times query the same series, so a generated run is conditioned on
the market path: the real one (what the market did), a frozen one (no news after the start),
or a scenario (e.g. QQQ -1% over 30 s: a headline) - see Market.scenario().
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
from numba import njit

ETFS = ("QQQ", "SMH")
CLOCK_TAUS_S = (0.1, 1.0, 10.0, 100.0)
VOL_TAUS_S = (1.0, 30.0)
N_ETF_FEAT = len(CLOCK_TAUS_S) + len(VOL_TAUS_S)
N_OWN = len(CLOCK_TAUS_S)
N_MKT = N_OWN + N_ETF_FEAT * len(ETFS)
NS = 1_000_000_000


# ----------------------------------------------------------------------------- extraction

def bbo_series(filtered_path: str | Path, ticker: str, start_s: int = 32_400,
               end_s: int = 57_600) -> dict[str, np.ndarray]:
    """Best bid / ask after every change, from a filtered ITCH stream (prices x 10000).
    Starts at 09:00 so that the averages are warm at the open."""
    from .book import OrderBook
    from .itch import load_stream
    from .lobster import Report, _replay

    sym = ticker.encode().ljust(8, b" ")
    book = OrderBook()
    t_out, bid_out, ask_out = [], [], []
    last = [None, None]
    lo, hi = start_s * NS, end_s * NS

    def emit(ts, *_):
        if not lo <= ts < hi:
            return
        b, a = book.best_bid(), book.best_ask()
        if (b, a) != (last[0], last[1]):
            last[0], last[1] = b, a
            t_out.append(ts)
            bid_out.append(-1 if b is None else b)
            ask_out.append(-1 if a is None else a)

    rep = Report(ticker=ticker, date="")
    _replay(load_stream(filtered_path), sym, book, emit, rep)
    if rep.locate is None:
        raise ValueError(f"{ticker} not in {filtered_path}")
    return {"t": np.array(t_out, dtype=np.int64), "bid": np.array(bid_out, dtype=np.int64),
            "ask": np.array(ask_out, dtype=np.int64)}


def save_bbo(s: dict, path: str | Path) -> None:
    np.savez_compressed(path, t=s["t"], bid=s["bid"], ask=s["ask"])


# ----------------------------------------------------------------------------- features

@njit(cache=True)
def _precompute(t, x, taus_ns, vtaus_ns, E, R):
    """E[k, j]: EMA_tau_j of the piecewise-constant x at time t[k] (just before x[k] takes
    over). R[k, j]: decayed sum of |jumps| up to and including the jump at t[k], in bps."""
    n = t.shape[0]
    for j in range(taus_ns.shape[0]):
        E[0, j] = x[0]
    for j in range(vtaus_ns.shape[0]):
        R[0, j] = 0.0
    for k in range(1, n):
        dt = float(t[k] - t[k - 1])
        for j in range(taus_ns.shape[0]):
            a = np.exp(-dt / taus_ns[j])
            E[k, j] = a * E[k - 1, j] + (1.0 - a) * x[k - 1]
        jump = abs(x[k] - x[k - 1]) * 1e4
        for j in range(vtaus_ns.shape[0]):
            R[k, j] = R[k - 1, j] * np.exp(-dt / vtaus_ns[j]) + jump


class Series:
    """One ETF's log mid as a step function of time, with what the feature lookups need.

    Scenarios do not copy the (multi-million point) arrays: frozen() stops the path at t0 (the
    averages keep decaying towards the last value, exactly as for a quote that never changes
    again) and shifted() adds a small overlay path. Averages are linear, so the overlay's
    deviations add; jumps at different times add in the turbulence sum."""

    def __init__(self, t: np.ndarray, logmid: np.ndarray) -> None:
        order = np.argsort(t, kind="stable")
        self.t = np.asarray(t, dtype=np.int64)[order]
        self.x = np.asarray(logmid, dtype=np.float64)[order]
        if len(self.t) == 0:
            raise ValueError("empty series")
        self.taus = np.array(CLOCK_TAUS_S) * NS
        self.vtaus = np.array(VOL_TAUS_S) * NS
        self.E = np.empty((len(self.t), len(self.taus)))
        self.R = np.empty((len(self.t), len(self.vtaus)))
        _precompute(self.t, self.x, self.taus, self.vtaus, self.E, self.R)
        self.t_end: int | None = None          # frozen after this time
        self.overlays: list["Series"] = []

    @classmethod
    def from_bbo(cls, s: dict) -> "Series":
        ok = (s["bid"] > 0) & (s["ask"] > 0) & (s["ask"] > s["bid"])
        mid = np.where(ok, (s["bid"] + s["ask"]) / 2.0, np.nan)
        # carry the last good quote over one-sided or crossed moments
        idx = np.where(ok, np.arange(len(mid)), -1)
        np.maximum.accumulate(idx, out=idx)
        keep = idx >= 0
        t, m = s["t"][keep], mid[idx[keep]]
        # one quote per timestamp, the last: a replace ('U') is a delete then an add at the same
        # ns, and the book between the two is not a quote anyone saw
        last = np.append(t[1:] != t[:-1], True)
        t, m = t[last], m[last]
        change = np.concatenate([[True], np.diff(m) != 0])
        return cls(t[change], np.log(m[change]))

    def _copy(self) -> "Series":
        c = object.__new__(Series)
        c.__dict__.update(self.__dict__)
        c.overlays = list(self.overlays)
        return c

    def _index(self, q: np.ndarray) -> np.ndarray:
        qc = q if self.t_end is None else np.minimum(q, self.t_end)
        return np.searchsorted(self.t, qc, side="right") - 1

    def value(self, q) -> np.ndarray:
        q = np.asarray(q, dtype=np.int64)
        k = self._index(q)
        v = np.where(k >= 0, self.x[np.maximum(k, 0)], self.x[0])
        for o in self.overlays:
            ko = o._index(q)
            v = v + np.where(ko >= 0, o.x[np.maximum(ko, 0)], 0.0)
        return v

    def _raw(self, q: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """(log mid - its averages [n, taus], turbulence sum in bps [n, vol taus]); 0 before
        the first point."""
        k = self._index(q)
        before = k < 0
        k = np.maximum(k, 0)
        dt = np.maximum(q - self.t[k], 0).astype(np.float64)
        a = np.exp(-dt[:, None] / self.taus[None, :])
        x = self.x[k][:, None]
        dev = x - (a * self.E[k] + (1 - a) * x)
        R = self.R[k] * np.exp(-dt[:, None] / self.vtaus[None, :])
        dev[before] = 0.0
        R[before] = 0.0
        return dev, R

    def features(self, q: np.ndarray) -> np.ndarray:
        """[len(q), N_ETF_FEAT] at times q (ns since midnight)."""
        q = np.asarray(q, dtype=np.int64)
        dev, R = self._raw(q)
        for o in self.overlays:
            d2, r2 = o._raw(q)
            dev, R = dev + d2, R + r2
        return np.hstack([np.clip(dev * 1e4 / 10, -5, 5), np.log1p(R) / 3])

    def shifted(self, t0: int, bps: float, ramp_s: float, step_ns: int = 10_000_000) -> "Series":
        """This path with an extra move of `bps` that builds up linearly (in 10 ms steps) over
        ramp_s seconds from t0 and then stays: a news shock on top of what the market did."""
        n = max(int(ramp_s * NS) // step_ns, 1)
        j = np.arange(n + 1)
        ov = Series(t0 + j * step_ns, bps / 1e4 * j / n)
        ov.t_end = self.t_end
        c = self._copy()
        c.overlays.append(ov)
        return c

    def frozen(self, t0: int) -> "Series":
        """The path up to t0, then flat: a market with no news after t0."""
        c = self._copy()
        c.t_end = t0 if self.t_end is None else min(t0, self.t_end)
        for i, o in enumerate(c.overlays):
            c.overlays[i] = o._copy()
            c.overlays[i].t_end = c.t_end
        return c


class Market:
    """The ETF series of one day, in ETFS order; features() gives the ETF part of the input."""

    def __init__(self, series: list[Series], date: str = "") -> None:
        if len(series) != len(ETFS):
            raise ValueError(f"need one series per ETF {ETFS}")
        self.series, self.date = series, date

    @classmethod
    def load(cls, date: str, root: str | Path = "data/market") -> "Market":
        root = Path(root)
        out = []
        for e in ETFS:
            p = root / f"{e}_{date}.npz"
            if not p.exists():
                raise FileNotFoundError(f"{p} missing: run notebooks/fetch_itch_colab.ipynb "
                                        "(market part) and download MyDrive/lobgen/market/")
            z = np.load(p)
            out.append(Series.from_bbo({k: z[k] for k in ("t", "bid", "ask")}))
        return cls(out, date)

    def features(self, q: np.ndarray) -> np.ndarray:
        return np.hstack([s.features(q) for s in self.series])

    def scenario(self, t0: int, bps: float, ramp_s: float = 30.0, etfs=ETFS) -> "Market":
        """News scenario: the chosen ETFs move an extra `bps` over ramp_s seconds from t0."""
        return Market([s.shifted(t0, bps, ramp_s) if e in etfs else s
                       for e, s in zip(ETFS, self.series)], self.date)

    def frozen(self, t0: int) -> "Market":
        return Market([s.frozen(t0) for s in self.series], self.date)

    def ret_bps(self, t0: int, t1: int, etf: str = "QQQ") -> float:
        s = self.series[ETFS.index(etf)]
        return float((s.value(np.array([t1]))[0] - s.value(np.array([t0]))[0]) * 1e4)


# ----------------------------------------------------------------------------- own clock EMA

@njit(cache=True)
def own_scan(t, mid, taus_ns, E, state, out):
    """Clock-time deviations of the stock's own log mid, in place.

    state = [last log mid (nan before the first), last time]; E [n_tau]. mid: forward-filled
    mid after each event, ticks (nan until the first). out[i] after event i (bps / 10)."""
    n = t.shape[0]
    T = taus_ns.shape[0]
    for i in range(n):
        x_prev = state[0]
        if not np.isnan(x_prev):
            dt = float(t[i]) - state[1]
            if dt > 0:
                for j in range(T):
                    a = np.exp(-dt / taus_ns[j])
                    E[j] = a * E[j] + (1.0 - a) * x_prev
        if not np.isnan(mid[i]):
            x = np.log(mid[i])
            if np.isnan(x_prev):
                for j in range(T):
                    E[j] = x
            state[0] = x
        state[1] = float(t[i])
        for j in range(T):
            d = 0.0
            if not np.isnan(state[0]):
                d = (state[0] - E[j]) * 1e4 / 10
            out[i, j] = min(max(d, -5.0), 5.0)


OWN_TAUS_NS = np.array(CLOCK_TAUS_S) * NS

In [ ]:
# Speed check: 15 s of one file. If this is under ~5 MB/s, total time is large; tell Claude.
import subprocess
r = subprocess.run(["curl", "-s", "-m", "15", "-o", "/dev/null", "-w", "%{speed_download}",
                    "-r", "0-2000000000", BASE + DAYS["2025-12-12"]], capture_output=True, text=True)
mbps = float(r.stdout or 0) / 1e6
total_gb = 51
print(f"{mbps:.1f} MB/s  ->  ~{total_gb * 1000 / max(mbps, 0.01) / 60:.0f} min for all 5 days")

In [ ]:
import gzip, json, shutil, subprocess, sys, time
sys.path.insert(0, "/content/pkg")
from lobgen.itch import filter_file

def content_length(url):
    h = subprocess.run(["curl", "-sI", url], capture_output=True, text=True).stdout
    return int([l for l in h.splitlines() if l.lower().startswith("content-length")][-1].split(":")[1])

def filter_to(raw, tickers, final, date, fname, size, t0, t1):
    local = f"/content/{date}.itch"
    rep = filter_file(raw, tickers, local)
    t2 = time.time()
    with open(local, "rb") as src, gzip.open(final + ".part", "wb") as dst:
        shutil.copyfileobj(src, dst)
    os.rename(final + ".part", final)
    rep.update(date=date, file=fname, tickers=tickers, compressed_bytes=size,
               download_min=(t1 - t0) / 60, filter_min=(t2 - t1) / 60)
    with open(final.replace(".itch.gz", ".filter.json"), "w") as f:
        json.dump(rep, f, indent=2)
    os.remove(local)
    print(date, tickers, f"filter {rep['filter_min']:.1f} min,",
          f"kept {rep['kept_bytes'] / 1e6:.1f} MB of {rep['raw_bytes'] / 1e9:.1f} GB", flush=True)

for date, fname in DAYS.items():
    jobs = [(t, f) for t, f in ((TICKERS, f"{OUT}/{date}.itch.gz"), (MARKET, f"{OUT_MKT}/{date}.itch.gz"))
            if not os.path.exists(f)]
    if not jobs:
        print(date, "already done"); continue
    url, raw = BASE + fname, "/content/raw.gz"
    size = content_length(url)
    t0 = time.time()
    # curl's own progress bar doesn't reach the cell output, so poll the file size.
    p = subprocess.Popen(["curl", "-sS", "-C", "-", "--retry", "20", "--retry-delay", "5",
                          "--retry-all-errors", "-o", raw, url])
    while p.poll() is None:
        time.sleep(30)
        have = os.path.getsize(raw) if os.path.exists(raw) else 0
        rate = have / max(time.time() - t0, 1) / 1e6
        print(f"  {date}: {have / 1e9:5.2f} / {size / 1e9:.2f} GB  ({rate:.0f} MB/s)", flush=True)
    if p.returncode:
        raise RuntimeError(f"{date}: curl exited {p.returncode}")
    got = os.path.getsize(raw)
    assert got == size, f"{date}: downloaded {got} of {size} bytes"
    t1 = time.time()
    print(date, f"download {(t1 - t0) / 60:.1f} min", flush=True)
    for tickers, final in jobs:
        filter_to(raw, tickers, final, date, fname, size, t0, t1)
    os.remove(raw)
print("filtering done")

In [ ]:
# Best bid / ask series of each ETF (what the model reads as "the market"), ~1-5 min per ETF-day.
import numpy as np
from lobgen.market import ETFS, bbo_series, save_bbo
assert tuple(MARKET) == ETFS, (MARKET, ETFS)
for date in DAYS:
    for etf in MARKET:
        out = f"{MKT}/{etf}_{date}.npz"
        if os.path.exists(out):
            print(etf, date, "already done"); continue
        t0 = time.time()
        s = bbo_series(f"{OUT_MKT}/{date}.itch.gz", etf)
        save_bbo(s, "/content/tmp.npz")
        shutil.copy("/content/tmp.npz", out)
        spread = (s["ask"] - s["bid"])[(s["ask"] > 0) & (s["bid"] > 0)] / 100
        print(etf, date, f"{len(s['t']):,} quote changes, median spread {np.median(spread):.1f} ticks,",
              f"{os.path.getsize(out) / 1e6:.1f} MB, {(time.time() - t0) / 60:.1f} min", flush=True)
print("done: download MyDrive/lobgen/market/ to data/market/ on the laptop")